# Basic RAG System

Build a Retrieval-Augmented Generation system from scratch.

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import AutoTokenizer, AutoModelForCausalLM

## 1. Document Chunking

In [ ]:
def chunk_documents(documents, chunk_size=512, overlap=50):
    chunks = []
    for doc in documents:
        words = doc.split()
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            chunks.append(chunk)
    return chunks

documents = ["Large Language Models (LLMs) are neural networks trained on vast amounts of text data. They can generate human-like text, answer questions, and perform various language tasks. Examples include GPT-4, Claude, and LLaMA."]
chunks = chunk_documents(documents)
print(f'Created {len(chunks)} chunks')

## 2. Embedding & Vector Store

In [ ]:
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5')
client = chromadb.Client()
collection = client.create_collection('rag_collection')
embeddings = embedder.encode(chunks)
collection.add(embeddings=embeddings.tolist(), documents=chunks, ids=[f'chunk_{i}' for i in range(len(chunks))])
print('Documents indexed successfully!')

## 3. Retrieval

In [ ]:
def retrieve(query, top_k=3):
    query_embedding = embedder.encode([query])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=top_k)
    return results['documents'][0]

query = 'What are LLMs?'
retrieved_docs = retrieve(query)
print('Retrieved documents:')
for i, doc in enumerate(retrieved_docs):
    print(f'{i+1}. {doc[:100]}...')

## 4. Generation with Context

In [ ]:
def generate_answer(query, retrieved_docs):
    context = '\n\n'.join(retrieved_docs)
    prompt = f'Based on the following context, answer the question:\n\nContext:\n{context}\n\nQuestion: {query}'
    print('Prompt created:')
    print(prompt[:500] + '...' if len(prompt) > 500 else prompt)
    return prompt

answer = generate_answer(query, retrieved_docs)